In [ ]:
import requests
import json
from typing import List, Dict
import PyPDF2 # 導入 PyPDF2 函式庫
import os # 導入 os 模組用於檢查檔案路徑

class OllamaChatbot:
    def __init__(self, model_name="llama3:8b", base_url="http://localhost:11434"):
        """
        初始化聊天機器人
        
        Args:
            model_name: Ollama模型名稱 (預設: llama3:8b)
            base_url: Ollama服務的基礎URL (預設: http://localhost:11434)
        """
        self.model_name = model_name
        self.base_url = base_url
        self.chat_url = f"{base_url}/api/chat"
        self.conversation_history: List[Dict[str, str]] = []
        self.attached_pdf_content: str = "" # 新增一個屬性來儲存附加的 PDF 內容
        self.pdf_attached_flag: bool = False # 標記是否有 PDF 內容被附加

    def _extract_text_from_pdf(self, pdf_path: str) -> str:
        """
        從 PDF 檔案中提取文字內容。
        這是內部使用的輔助方法。
        """
        text = ""
        try:
            with open(pdf_path, 'rb') as file:
                reader = PyPDF2.PdfReader(file)
                for page_num in range(len(reader.pages)):
                    page = reader.pages[page_num]
                    # 嘗試提取文字，如果頁面沒有文字內容則跳過
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + "\n" # 每頁之間加一個換行符
        except PyPDF2.errors.PdfReadError:
            return f"錯誤: 無法讀取 PDF 檔案 '{pdf_path}'，可能檔案已損壞或加密。"
        except FileNotFoundError:
            return f"錯誤: 檔案不存在 '{pdf_path}'。"
        except Exception as e:
            return f"提取 PDF 文字時發生未知錯誤: {e}"
        return text

    def attach_pdf(self, pdf_path: str) -> str:
        """
        附加一個 PDF 檔案，提取其內容並儲存以供後續對話使用。
        """
        if not os.path.exists(pdf_path):
            return f"錯誤: 指定的 PDF 檔案不存在: '{pdf_path}'"
        if not pdf_path.lower().endswith('.pdf'):
            return f"錯誤: 檔案 '{pdf_path}' 不是一個 PDF 檔案。"

        print(f"📄 正在從 '{pdf_path}' 提取文字內容，請稍候...")
        extracted_text = self._extract_text_from_pdf(pdf_path)

        if extracted_text.startswith("錯誤:"):
            self.attached_pdf_content = ""
            self.pdf_attached_flag = False
            return extracted_text # 返回錯誤訊息
        elif not extracted_text.strip():
            self.attached_pdf_content = ""
            self.pdf_attached_flag = False
            return "⚠️ 警告: 從 PDF 中提取的文字內容為空。請檢查 PDF 檔案是否包含可讀文字。"
        else:
            self.attached_pdf_content = extracted_text
            self.pdf_attached_flag = True
            # 可選：將 PDF 內容作為系統訊息添加到對話歷史的開頭，這樣模型在每次對話時都會看到它
            # 但更好的做法是在每次發送訊息時，將其作為一個特殊的上下文訊息傳遞
            # self.conversation_history.insert(0, {"role": "system", "content": f"以下是參考文件內容：\n{self.attached_pdf_content}"})
            base, show_len = 0, 0
            print(f"📖 提取內容前 {show_len} 字：\n{extracted_text[base:base+show_len]}...\n" + "-"*30)
            return f"✅ PDF 檔案 '{pdf_path}' 的內容已成功附加為對話上下文。"

    def send_message(self, message: str) -> str:
        """
        發送訊息給LLM並獲取回應。
        如果已附加 PDF，則將 PDF 內容作為上下文傳遞。
        
        Args:
            message: 用戶輸入的訊息
            
        Returns:
            LLM的回應
        """
        # 準備要發送給 Ollama 的訊息列表
        messages_to_send = []

        # 如果有附加 PDF 內容，將其作為系統提示添加到訊息列表的開頭
        if self.pdf_attached_flag and self.attached_pdf_content:
            # 這裡將 PDF 內容作為一個獨立的系統訊息傳遞，而不是直接修改用戶訊息
            # 這樣可以更好地區分用戶輸入和文件上下文
            messages_to_send.append({
                "role": "system",
                "content": f"以下是參考文件內容：\n\n{self.attached_pdf_content}\n\n請根據這些內容來回答問題，如果問題與文件無關，請說明。"
            })
            
        # 將當前的對話歷史添加到訊息列表
        messages_to_send.extend(self.conversation_history)
        
        # 將用戶當前訊息添加到訊息列表
        messages_to_send.append({"role": "user", "content": message})
        
        # 準備請求資料
        payload = {
            "model": self.model_name,
            "messages": messages_to_send, # 使用包含 PDF 上下文和歷史的訊息列表
            "stream": False,
            "options": {
                "num_ctx": 16384  # 強制將上下文視窗放大到 16k，確保塞得下整份簡章
            }
        }
        
        try:
            # 發送請求到Ollama
            response = requests.post(
                self.chat_url,
                json=payload,
                headers={"Content-Type": "application/json"},
                timeout=None 
            )
            
            if response.status_code == 200:
                result = response.json()
                assistant_message = result["message"]["content"]
                
                # 將用戶訊息和助手回應添加到對話歷史（僅用於顯示和後續對話記憶）
                self.conversation_history.append({"role": "user", "content": message})
                self.conversation_history.append({
                    "role": "assistant", 
                    "content": assistant_message
                })
                
                return assistant_message
            else:
                return f"錯誤: HTTP {response.status_code} - {response.text}"
                
        except requests.exceptions.RequestException as e:
            return f"連接錯誤: {str(e)}"
        except json.JSONDecodeError as e:
            return f"JSON解析錯誤: {str(e)}"
        except Exception as e:
            return f"未知錯誤: {str(e)}"
    
    def new_session(self):
        """
        重置對話記憶，開始新的對話會話，並清除附加的 PDF 內容。
        """
        self.conversation_history = []
        self.attached_pdf_content = "" # 清除附加的 PDF 內容
        self.pdf_attached_flag = False # 重置標記
        print("✅ 對話記憶已重置，開始新的會話！")
    
    def show_history(self):
        """
        顯示當前對話歷史
        """
        if not self.conversation_history:
            print("📝 目前沒有對話記錄")
            return
            
        print("\n📚 對話歷史:")
        print("-" * 50)
        for i, msg in enumerate(self.conversation_history, 1):
            role = "用戶" if msg["role"] == "user" else "助手"
            print(f"{i}. {role}: {msg['content']}")
        print("-" * 50)
    
    def check_model(self) -> bool:
        """
        檢查模型是否可用
        
        Returns:
            模型是否可用
        """
        try:
            models_url = f"{self.base_url}/api/tags"
            response = requests.get(models_url, timeout=10)
            
            if response.status_code == 200:
                models = response.json().get("models", [])
                available_models = [model["name"] for model in models]
                
                if self.model_name in available_models:
                    print(f"✅ 模型 {self.model_name} 已就緒")
                    return True
                else:
                    print(f"❌ 模型 {self.model_name} 不可用")
                    print(f"可用模型: {available_models}")
                    return False
            else:
                print(f"❌ 無法連接到Ollama服務: HTTP {response.status_code}")
                return False
                
        except Exception as e:
            print(f"❌ 檢查模型時發生錯誤: {str(e)}")
            return False

def main(model_name="qwen3:14b"):
    """
    主要執行函model_name數
    """
    print(f"🤖 Ollama {model_name} 聊天機器人")
    print("=" * 50)
    
    # 初始化聊天機器人
    chatbot = OllamaChatbot(model_name)
    
    # 檢查模型可用性
    if not chatbot.check_model():
        print("\n請確保:")
        print("1. Ollama服務正在運行 (ollama serve)")
        print("2. 已下載llama3模型 (ollama pull llama3)")
        return
    
    print("\n使用說明:")
    print("• 直接輸入訊息與AI對話")
    print("• 輸入 'New Session' 重置對話記憶並清除附加的 PDF")
    print("• 輸入 'history' 查看對話歷史")
    print("• 輸入 'attach_pdf <檔案路徑>' 附加 PDF 檔案作為上下文 (例如: attach_pdf my_document.pdf)")
    print("• 輸入 'quit' 或 'exit' 退出程式")
    print("-" * 50)
    
    # 主對話迴圈
    while True:
        try:
            user_input = input("\n👤 您: ").strip()
            
            # 退出指令
            if user_input.lower() in ['quit', 'exit', '退出', 'q']:
                print("👋 再見！")
                break
            
            # 空輸入處理
            if not user_input:
                print("⚠️  請輸入訊息")
                continue
            
            # 新會話指令
            if user_input.lower() == 'new session':
                chatbot.new_session()
                continue
            
            # 查看歷史指令
            if user_input.lower() == 'history':
                chatbot.show_history()
                continue
            
            # 附加 PDF 指令
            if user_input.lower().startswith('attach_pdf '):
                pdf_path = user_input[len('attach_pdf '):].strip()
                result_message = chatbot.attach_pdf(pdf_path)
                print(result_message)
                continue # 處理完 PDF 後繼續等待用戶輸入
            
            # 發送訊息並獲取回應
            print("🤔 思考中...")
            response = chatbot.send_message(user_input)
            print(f"🤖 助手: {response}")
            
        except KeyboardInterrupt:
            print("\n\n👋 程式已中斷，再見！")
            break
        except Exception as e:
            print(f"❌ 發生錯誤: {str(e)}")

if __name__ == "__main__":
    main(model_name="qwen-cpu-3-30b:latest")

🤖 Ollama qwen-cpu-3-30b:latest 聊天機器人
✅ 模型 qwen-cpu-3-30b:latest 已就緒

使用說明:
• 直接輸入訊息與AI對話
• 輸入 'New Session' 重置對話記憶並清除附加的 PDF
• 輸入 'history' 查看對話歷史
• 輸入 'attach_pdf <檔案路徑>' 附加 PDF 檔案作為上下文 (例如: attach_pdf my_document.pdf)
• 輸入 'quit' 或 'exit' 退出程式
--------------------------------------------------



👤 您:  attach_pdf 114選課說明.pdf


📄 正在從 '114選課說明.pdf' 提取文字內容，請稍候...
📖 提取內容前 0 字：
...
------------------------------
✅ PDF 檔案 '114選課說明.pdf' 的內容已成功附加為對話上下文。



👤 您:  初選前該注意什麼? 以繁體中文回答


🤔 思考中...
